# Ejercicio 2. RAG industrial con manuales técnicos

**Enunciado:** Descargar varios manuales técnicos de máquinas de una planta y desarrollar un chatbot industrial con RAG. Se sugiere Ollama.

Fuente: Martínez P. (2026), sección 7.11.

## Cómo utilizar este notebook

Abre el archivo en Jupyter o VS Code con un kernel Python del mismo entorno del proyecto. Colócalo en la raíz o en `notebooks/`. Las celdas de texto contienen la respuesta al ejercicio; las de código generan evidencia al ejecutarlas en tu computador.

No se han ejecutado las consultas contra tu Ollama ni tu base de datos. Los resultados esperados no se presentan como resultados obtenidos. Si hace falta instalar Jupyter, utiliza en tu entorno `python -m pip install notebook ipykernel`; las dependencias de la aplicación se instalan desde `requirements.txt`.

Las rutas y funciones corresponden al código desarrollado en la conversación. Los manuales concretos, los modelos instalados y el contenido real de tus skills deben verificarse localmente.

## Respuesta

Se desarrolló un flujo RAG —generación aumentada por recuperación— para responder preguntas a partir de manuales industriales. Los PDF se colocan en `data/manuals/` y se indexan explícitamente. Se deben seleccionar documentos oficiales del fabricante y del modelo consultado.

### Funcionamiento y archivos

| Etapa | Archivo | Responsabilidad |
|---|---|---|
| Configuración | `config.py` | Define servidor, modelos y rutas absolutas del proyecto |
| Lectura | `rag/document_loader.py` | Extrae texto de cada página PDF y conserva archivo y página física |
| División | `rag/chunker.py` | Divide cada página en fragmentos de 1.000 caracteres con 150 de solapamiento, ajustables |
| Embeddings | `rag/embeddings.py` | Envía los textos a Ollama y valida los vectores recibidos |
| Almacenamiento | `rag/vector_store.py` | Guarda texto, vectores y metadatos en una colección persistente de ChromaDB |
| Indexación | `rag/index_documents.py` | Coordina las etapas y elimina los fragmentos anteriores sobrantes del mismo archivo |
| Recuperación | `rag/retriever.py` | Convierte la pregunta en vector y recupera fragmentos cercanos |
| Respuesta | `rag/rag_chat.py` | Envía la pregunta y los textos recuperados al modelo, solicitando citas `[F1]`, `[F2]`, etc. |

Ollama recibe textos en `/api/embed` y devuelve sus embeddings (Ollama, s. f.-a). Se utiliza el mismo modelo de embeddings para documentos y preguntas. El nombre predeterminado propuesto es `embeddinggemma`; el modelo conversacional se configura por separado en `LLM_MODEL`.

ChromaDB se configura con distancia coseno para comparar vectores (Chroma, s. f.-a). Una distancia menor significa mayor cercanía en esa representación, no un porcentaje de certeza.

Una precisión conceptual: el índice almacena un vector por fragmento en esta implementación. El modelo generativo recibe **el texto** de los fragmentos recuperados junto con la pregunta, no los vectores de ChromaDB directamente.

### Ejemplo de ejecución

Reemplazar `manual_bomba.pdf` por el nombre real del manual disponible:

```bash
python -m rag.index_documents "data/manuals/manual_bomba.pdf"
python -m rag.retriever "¿Qué mantenimiento indica el manual?"
python -m rag.rag_chat "¿Qué mantenimiento indica el manual?"
```

Para la demostración del ejercicio deben indexarse varios manuales. Con uno solo se puede probar el flujo, pero no se satisface literalmente la solicitud de trabajar con varios documentos.

### Límites

- El lector implementado admite PDF, no Markdown ni páginas web.
- `pypdf` extrae texto existente; no realiza OCR de imágenes escaneadas. También pueden perderse relaciones entre columnas y celdas (pypdf, s. f.).
- Los cortes por caracteres pueden dividir palabras u oraciones y no unen páginas consecutivas.
- Las citas son solicitadas al modelo, pero su contenido no se verifica automáticamente.
- Una recuperación insuficiente no demuestra que la información esté ausente de todo el manual.
- La reindexación no es atómica y mantiene los vectores del documento en memoria. Debe ejecutarse sin indexaciones simultáneas en esta versión.

## Demostración en tu proyecto

In [ ]:
from pathlib import Path
import sys

# Coloca los notebooks en la raíz del proyecto o en su carpeta notebooks/.
# Si están en otro lugar, escribe aquí la ruta de tu proyecto.
PROJECT_DIR = None  # Ejemplo: Path(r"C:\industrial-ai-assistant")

if PROJECT_DIR is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_DIR = next(
        (p for p in candidates if (p / "config.py").is_file()
         and (p / "app.py").is_file()), None
    )
if PROJECT_DIR is None:
    raise FileNotFoundError("Indica PROJECT_DIR con la ruta de tu proyecto.")
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
if not (PROJECT_DIR / "config.py").is_file():
    raise FileNotFoundError("La ruta indicada no contiene config.py.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Proyecto:", PROJECT_DIR)

In [ ]:
PDF_DIR = PROJECT_DIR / "data" / "manuals"
pdfs = sorted(p for p in PDF_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
for p in pdfs:
    print(p.relative_to(PROJECT_DIR))
print("PDF disponibles:", len(pdfs))

Selecciona un PDF de la lista. Para indexar varios manuales, repite las celdas con cada archivo. La indexación escribe en ChromaDB y puede actualizar registros del mismo documento.

In [ ]:
PDF_PATH = None  # Sustituye por PROJECT_DIR / "data" / ... / "archivo.pdf"
RUN_INDEX = False  # Cambia a True únicamente cuando hayas seleccionado el archivo.

if RUN_INDEX:
    if PDF_PATH is None:
        raise ValueError("Selecciona PDF_PATH antes de indexar.")
    from rag.index_documents import index_pdf
    summary = index_pdf(PDF_PATH)
    print(summary)
else:
    print("Indexación sin ejecutar.")

La consulta usa la colección compartida. Los resultados pueden incluir documentos de la otra carpeta; revisa las fuentes y no los presentes como corpus separados.

In [ ]:
RUN_QUERY = False  # Activa después de indexar y comprobar Ollama.
QUESTION = "¿Qué mantenimiento indica el manual?"

if RUN_QUERY:
    from rag.retriever import retrieve
    matches = retrieve(QUESTION, top_k=3)
    for number, match in enumerate(matches, 1):
        print(number, match["source"], "página", match["page"])
        print("Distancia:", match["distance"])
        print(match["text"], "\n")
else:
    print("Recuperación sin ejecutar.")

In [ ]:
if RUN_QUERY:
    from rag.rag_chat import answer_question
    result = answer_question(QUESTION, top_k=3)
    print(result["answer"])
    print("\nFuentes enviadas:")
    for source in result["sources"]:
        print(source)
else:
    print("Generación sin ejecutar.")

## Registro de verificación

| Prueba | Resultado real | Revisión |
|---|---|---|
| Pregunta con respuesta explícita | Pendiente | Comparar respuesta y página original |
| Misma pregunta reformulada | Pendiente | Evaluar si recupera información equivalente |
| Pregunta ajena al documento | Pendiente | Verificar reconocimiento de información insuficiente |
| Código o término exacto | Pendiente | Comprobar coincidencia y evitar confusiones |

**Conclusión:** la recuperación y generación forman un chatbot documental básico. Su calidad debe demostrarse con las fuentes reales, no inferirse del hecho de que el código ejecute.

## Referencias

AI Hero. (s. f.). *Become a real AI Hero*. https://www.aihero.dev/

Chroma. (s. f.-a). *Configure collections*. https://docs.trychroma.com/docs/collections/configure

Chroma. (s. f.-b). *Query and get*. https://docs.trychroma.com/docs/querying-collections/query-and-get

Espressif Systems. (s. f.). *Support for external RAM: ESP32-S3*. ESP-IDF Programming Guide. https://docs.espressif.com/projects/esp-idf/en/stable/esp32s3/api-guides/external-ram.html

Martínez P., J. J. (2026, septiembre). *7. Inteligencia artificial generativa* [Material de curso proporcionado en el archivo `7.md`].

Ollama. (s. f.-a). *Generate embeddings*. https://docs.ollama.com/api/embed

Ollama. (s. f.-b). *Tool calling*. https://docs.ollama.com/capabilities/tool-calling

pypdf. (s. f.). *Extract text from a PDF*. https://pypdf.readthedocs.io/en/stable/user/extract-text.html